خیلی خب، بیا قدم‌به‌قدم بریم سراغ دنیای تولید متن (Text Generation) با پایتورچ، مخصوصاً برای ادامه‌دادن یک بیت شعر فارسی به سبک یک شاعر خاص. قراره از پایه تا پیشرفته رو پوشش بدیم: از آماده‌سازی داده و توکن‌سازی تا مدل‌های LSTM و ترنسفورمر، آموزش، و روش‌های مختلف تولید (Sampling Strategies) مثل temperature، top-k، top-p و beam search. آخرش هم یه راهکار سریع با مدل‌های آماده می‌گم که پروژه‌ات رو سریع‌تر جواب بده.

---

۱. مدل‌سازی زبان (Language Modeling) یعنی چی؟

هدف ما ساختن مدلی است که با دیدن یک توالی از واژه‌ها (یا کاراکترها)، احتمال واژه (یا کاراکتر) بعدی را پیش‌بینی کند. به این مدل، مدل زبانی (Language Model) می‌گویند. مدل‌های زبانی می‌تونن به‌صورت خودرگرسیو (Autoregressive) باشن: یعنی برای تولید گام‌به‌گام از خروجی‌های قبلی خودش استفاده می‌کنه. این دقیقاً همون چیزیه که ما برای ادامه‌دادن یک بیت شعر نیاز داریم.

شما یک بیت (مثلاً یک مصرع) رو به مدل می‌دید و مدل مصرع بعدی رو کلمه‌به‌کلمه (یا توکن‌به‌توکن) تولید می‌کنه.

---

۲. آماده‌سازی دیتاست شعر شاعر قدیمی

جمع‌آوری و تمیزکاری

فرض کنید اشعار شاعر (مثلاً حافظ، سعدی، مولانا) رو در یک فایل متنی داریم، هر بیت در یک خط. بهتره نشانه‌های اضافی مثل علائم نگارشی مدرن (نقطه، ویرگول) حذف بشن مگر اینکه بخواید مدل اون‌ها رو هم یاد بگیره. نیم‌فاصله‌ها و کاراکترهای فارسی رو یکسان‌سازی کنید:

· «ی» و «ي» عربی ← «ی»
· «ک» و «ك» عربی ← «ک»
· اعداد فارسی/انگلیسی یکسان‌سازی (یا حذف)
· حذف حرکات (اَ، اِ، اُ) معمولاً کمک می‌کنه مدل بهتر یاد بگیره (مگر شعر عروضی رو با اعراب‌گذاری بخواید).

یک تابع تمیزکاری ساده:

```python
import re

def clean_persian(text):
    text = text.replace('ي', 'ی').replace('ك', 'ک')
    text = re.sub(r'[ًٌٍَُِّْ]', '', text)  # حذف اعراب
    # نیم‌فاصله استاندارد
    text = text.replace('\u200c', ' ')  # یا نگه داشتن نیم‌فاصله بستگی به توکنایزر دارد
    return text
```

توکن‌سازی (Tokenization)

سه رویکرد اصلی داریم:

۱. سطح کاراکتر (Character-level): هر کاراکتر یک توکن. دایره لغات کوچک (حدود ۵۰-۶۰ تا برای فارسی)، ساده، اما توالی‌ها طولانی‌تر می‌شن و مدل باید وابستگی‌های بلندمدت یاد بگیره. برای شروع عالیه.

۲. سطح واژه (Word-level): هر واژه یک توکن. دایره واژگان بزرگ (مثلاً ۵۰ هزارتا)، مشکل کلمات خارج از دایره لغات (OOV). برای شعر فارسی که دایره لغات محدودتره و کلمات تکراری زیاده، با یک دایره واژگان مناسب می‌تونه خوب جواب بده.

۳. سطح زیرواژه (Subword) مثل BPE یا SentencePiece: بهترین گزینه برای پروژه‌های واقعی. دایره واژگان متوسط (مثلاً ۵۰۰۰ تا ۲۰۰۰۰)، کلمات ناشناخته رو به قطعات کوچک‌تر می‌شکنه. کتابخانه tokenizers (HuggingFace) یا sentencepiece عالی کار می‌کنن.

توصیه: برای آموزش از پایه، با سطح کاراکتر شروع کنیم چون پیاده‌سازی راحت‌تره و همه چیز شفافه. بعداً می‌تونید به BPE ارتقا بدید.

توکنایزر کاراکتری ساده:

```python
class CharTokenizer:
    def __init__(self, texts):
        chars = sorted(set(''.join(texts)))
        self.stoi = {ch: i+1 for i, ch in enumerate(chars)}  # 0 برای padding
        self.itos = {i+1: ch for i, ch in enumerate(chars)}
        self.vocab_size = len(chars) + 1

    def encode(self, text):
        return [self.stoi[ch] for ch in text if ch in self.stoi]

    def decode(self, indices):
        return ''.join([self.itos.get(i, '') for i in indices if i != 0])
```

ساخت Dataset و DataLoader

مدل نیاز داره که از یک توالی x (مثلاً ۱۰۰ کاراکتر)، کاراکتر بعدی y رو پیش‌بینی کنه. با پنجره‌ی لغزان (Sliding Window) روی کل متن اشعار، جفت‌های (ورودی، هدف) می‌سازیم.

```python
import torch
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, text, seq_len, tokenizer):
        self.seq_len = seq_len
        self.data = tokenizer.encode(text)
        self.vocab_size = tokenizer.vocab_size

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = self.data[idx: idx + self.seq_len]
        y = self.data[idx + 1: idx + self.seq_len + 1]
        return torch.tensor(x), torch.tensor(y)
```

برای پَد کردن در batchها (اگر طول ثابت نباشد) می‌تونیم از collate_fn استفاده کنیم، اما وقتی seq_len ثابته و همه نمونه‌ها هم‌طولن، DataLoader پیش‌فرض کافیه:

```python
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)
```

---

۳. مدل‌های تولیدمتن با پایتورچ

مدل ۱: شبکه LSTM (ساده و کلاسیک)

```python
import torch.nn as nn

class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, hidden=None):
        # x shape: (batch, seq_len)
        emb = self.dropout(self.embedding(x))  # (batch, seq_len, embed_dim)
        out, hidden = self.lstm(emb, hidden)   # out: (batch, seq_len, hidden_dim)
        out = self.dropout(out)
        logits = self.fc(out)                  # (batch, seq_len, vocab_size)
        return logits, hidden
```

نکته مهم: hidden حالت پنهان LSTM هست که می‌تونیم در زمان تولید (inference) برای ادامه‌دادن نسل ازش استفاده کنیم.

مدل ۲: ترنسفورمر (Transformer) دیکودر-فقط (مثل GPT)

برای تولید متن، معماری Decoder-only با causal masking به کار می‌ره تا هر توکن فقط توکن‌های قبلی رو ببینه. پایتورچ nn.TransformerDecoderLayer رو داره، ولی پیاده‌سازی دستی Causal Mask هم ساده است.

نسخه مینیمال GPT با PyTorch:

```python
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4,
                 dim_feedforward=1024, dropout=0.2, max_len=512):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        decoder_layer = nn.TransformerDecoderLayer(d_model, nhead,
                                                   dim_feedforward, dropout,
                                                   batch_first=True)
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers)
        self.ln = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, vocab_size)
        self.max_len = max_len
        self.d_model = d_model

    def forward(self, x):
        # x: (batch, seq_len)
        batch_size, seq_len = x.shape
        # Causal mask: ماتریس مثلثی بالا-راست صفر شود
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool().to(x.device)
        pos = torch.arange(0, seq_len, device=x.device).unsqueeze(0)  # (1, seq_len)
        tok_emb = self.token_emb(x) * (self.d_model ** 0.5)
        pos_emb = self.pos_emb(pos)
        x_emb = tok_emb + pos_emb  # (batch, seq_len, d_model)

        # TransformerDecoder نیاز به memory ندارد (None) و tgt همان x_emb است
        out = self.transformer(tgt=x_emb, memory=None, tgt_mask=mask)
        out = self.ln(out)
        logits = self.fc(out)  # (batch, seq_len, vocab_size)
        return logits
```

توجه: tgt_mask باید True جایی باشد که باید پوشانده (نادیده گرفته) شود، یعنی آینده. بنابراین triu با قطر ۱ به ما ماسک مناسب را می‌دهد.

برای بهبود می‌توانید از nn.TransformerEncoder با src_mask و یک memory=None استفاده کنید (کلاس nn.Transformer کامل). اما کد بالا خوانا و کار راه‌اندازه.

---

۴. آموزش مدل (Training)

حلقه آموزش استاندارد

```python
def train(model, loader, optimizer, criterion, device, clip=1.0):
    model.train()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits, _ = model(x)  # برای LSTM که hidden خروجی دوم است.
        # برای GPT خروجی فقط logits است: logits = model(x)
        loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)
```

تنظیمات معمول:

```python
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = nn.CrossEntropyLoss(ignore_index=0)  # 0 پدینگ است
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.9)
```

اگر از LSTM استفاده می‌کنید، دقت کنید که hidden حالت را در طولbatchها حفظ نکنید (در هر batch صفرش کنید) مگر اینکه از تکنیک Truncated BPTT استفاده کنید که در این صورت hidden را جدا (detach) می‌کنید.

معیار سرگشتگی (Perplexity)

Perplexity = exp(loss). این عدد نشان می‌دهد که مدل به‌طور متوسط بین چند گزینه مردد است. هرچه کمتر بهتر. می‌تونید روی یک مجموعه اعتبارسنجی حساب کنید.

---

۵. تولید (Sampling) – ادامه دادن بیت ورودی

بعد از آموزش، یک بیت (مثلاً یک مصرع) به مدل می‌دهیم و بقیه را تولید می‌کنیم. فرآیند به این شکل است:

1. توکن‌های ورودی را به مدل بده.
2. logits آخرین موقعیت را بردار.
3. احتمال‌ها را با softmax حساب کن (و با temperature تنظیم کن).
4. یک توکن نمونه‌برداری (sample) کن و به توالی اضافه کن.
5. تکرار تا طول دلخواه.

روش‌های مختلف نمونه‌برداری

۱. حریصانه (Greedy): انتخاب توکن با بالاترین احتمال (سریع، تکراری و خسته‌کننده)

```python
def greedy_sample(logits):
    return torch.argmax(logits, dim=-1)
```

۲. نمونه‌برداری با دما (Temperature Sampling)

احتمالات را با تقسیم بر temperature نرم‌تر یا تیزتر می‌کنیم.

```python
def sample_with_temp(logits, temperature=0.8):
    probs = torch.softmax(logits / temperature, dim=-1)
    return torch.multinomial(probs, num_samples=1)
```

· دما < ۱: خروجی محافظه‌کارانه‌تر (تکراری‌تر)
· دما > ۱: خلاق‌تر ولی شاید بی‌ربط

۳. Top-K Sampling

فقط از بین K توکن با بالاترین احتمال انتخاب کن.

```python
def top_k_sample(logits, k=40):
    values, indices = torch.topk(logits, k, dim=-1)
    min_val = values[:, -1].unsqueeze(-1)
    # توکن‌های خارج از top-k را منفی‌بی‌نهایت کن
    logits[logits < min_val] = float('-inf')
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)
```

۴. Top-P (Nucleus Sampling)

مجموعه‌ای از توکن‌ها را انتخاب کن که مجموع احتمالشان به آستانه p برسد.

```python
def top_p_sample(logits, p=0.9):
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    sorted_probs = torch.softmax(sorted_logits, dim=-1)
    cum_probs = torch.cumsum(sorted_probs, dim=-1)
    # ماسک: توکن‌هایی که مجموع احتمال‌شان از p بیشتر است
    mask = cum_probs - sorted_probs > p  # True برای توکن‌های خارج از هسته
    sorted_logits[mask] = float('-inf')
    # بازگرداندن ترتیب اصلی
    logits = torch.zeros_like(logits).scatter_(-1, sorted_idx, sorted_logits)
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)
```

۵. Beam Search

برای تولید منسجم‌تر و با احتمال کلی بالاتر. چندین دنباله نامزد را همزمان نگه می‌داریم.
پیاده‌سازی کامل آن کمی طولانی‌ست، اما برای تولید شعر می‌تواند مفید باشد. ساختار کلی:

· تعداد beamها (مثلاً ۴) را مشخص کن.
· در هر گام برای هر beam، top-k توکن بعدی را در نظر بگیر و نمره (log-prob) را به‌روز کن.
· از بین beam*top-k نامزد، beam تا با بالاترین نمره را انتخاب کن.
· وقتی یک دنباله به توکن پایان (مثلاً سرخط) برسد، آن را تمام کن.
· در نهایت دنباله با بیشترین امتیاز را برگردان.

پیاده‌سازی از صفر ممکن است کد زیادتری بخواهد، می‌توانید از کتابخانه‌هایی مثل transformers استفاده کنید که beam search آماده دارد، ولی خودتان هم می‌توانید بنویسید (من برای خلوتر ماندن پاسخ، کد را در صورت نیاز بعداً می‌دهم).

تابع کامل تولید

```python
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=50, strategy='top_p',
             temperature=0.8, top_k=40, top_p=0.9, device='cpu'):
    model.eval()
    # تبدیل ورودی به ایندکس
    idx = torch.tensor([tokenizer.encode(prompt)], device=device)
    for _ in range(max_new_tokens):
        # اگر مدل ترنسفورمر باشه و طول حداکثر داره، ورودی رو برش بزن
        if hasattr(model, 'max_len'):
            idx_crop = idx[:, -model.max_len:]
        else:
            idx_crop = idx

        logits = model(idx_crop)
        # برای LSTM که خروجی (logits, hidden) است
        if isinstance(logits, tuple):
            logits = logits[0]
        logits = logits[:, -1, :]  # logits آخرین موقعیت

        if strategy == 'greedy':
            next_token = greedy_sample(logits)
        elif strategy == 'temperature':
            next_token = sample_with_temp(logits, temperature)
        elif strategy == 'top_k':
            next_token = top_k_sample(logits, top_k)
        elif strategy == 'top_p':
            next_token = top_p_sample(logits, top_p)
        else:
            raise ValueError("Unknown strategy")

        idx = torch.cat([idx, next_token], dim=-1)

    generated = tokenizer.decode(idx[0].tolist())
    return generated[len(prompt):]  # فقط ادامه را برگردان
```

---

۶. کل مسیر عملی (End-to-End)

خلاصه آنچه برای راه‌اندازی پروژه باید انجام دهید:

1. متن اشعار را تمیز و یکپارچه کن.
2. توکنایزر کاراکتری بساز (یا BPE آموزش بده).
3. Dataset و DataLoader با طول توالی مثلاً ۲۰۰.
4. مدل LSTM یا GPT را با vocab_size مناسب تعریف کن.
5. حلقه آموزش را با ذخیره مدل (checkpoint) هر چند epoch.
6. تولید با یکی از استراتژی‌ها.

مثال سریع:

```python
# ساخت توکنایزر
all_text = open('hafez.txt', encoding='utf-8').read()
tokenizer = CharTokenizer([all_text])

# دیتاست
dataset = TextDataset(all_text, seq_len=200, tokenizer=tokenizer)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

# مدل
model = LSTMLanguageModel(vocab_size=tokenizer.vocab_size,
                          embed_dim=256, hidden_dim=512).to(device)
# یا: model = GPTLanguageModel(vocab_size=tokenizer.vocab_size).to(device)

# آموزش (چند epoch)
for epoch in range(20):
    loss = train(model, loader, optimizer, criterion, device)
    scheduler.step()
    print(f"Epoch {epoch} loss: {loss:.3f}")

# تست تولید
print(generate(model, tokenizer, "الا یا ایها الساقی ادر کأساً و ناولها", max_new_tokens=80))
```

---

۷. نکات پیشرفته و بهبود کیفیت شعر

· توکنایزر BPE: خروجی‌های بسیار روان‌تری نسبت به سطح کاراکتر می‌دهد. از کتابخانه tokenizers استفاده کنید:
  ```python
  from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers
  tokenizer = Tokenizer(models.BPE())
  tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
  trainer = trainers.BpeTrainer(vocab_size=8000, special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"])
  tokenizer.train_from_iterator(poems_list, trainer)
  tokenizer.save("poet-tokenizer.json")
  ```
  سپس برای encode/decode با PyTorch می‌تونید از tokenizer.encode(...).ids استفاده کنید.
· مدل از پیش آموزش‌دیده (Pretrained): بهترین نتیجه عملی را مدل‌های زبانی بزرگ فارسی مثل bolbolzaban/gpt2-persian یا HooshvareLab/bert-fa-base-uncased (در حالت LM) می‌دهند. کافیست با HuggingFace Transformers روی دیتاست شعر fine-tune کنید:
  ```python
  from transformers import GPT2LMHeadModel, AutoTokenizer
  model = GPT2LMHeadModel.from_pretrained("bolbolzaban/gpt2-persian")
  tokenizer = AutoTokenizer.from_pretrained("bolbolzaban/gpt2-persian")
  # tokenizer.pad_token = tokenizer.eos_token
  # سپس Fine-tuning با Trainer یا حلقه دستی
  ```
  با این کار مدل وزن‌های زبان فارسی را بلد است و شعر شما فقط سبک را روی آن تنظیم می‌کند. خروجی‌ها بسیار شاعرانه‌تر خواهند شد.
· تنظیم طول و پدینگ: در شعر، مصرع‌ها طول متغیری دارند. بهتر است ابیات را با یک توکن جداکننده مثل \n تفکیک کنید و مدل یاد بگیرد که end-of-line را تولید کند. در تولید، وقتی مدل توکن خط جدید تولید کرد، بیت تمام می‌شود.
· اضافه کردن Embedding وزن و قافیه (اختیاری): می‌توانید با برچسب‌گذاری وزن عروضی به هر مصرع، یک Embedding وزن به مدل بدهید تا ریتم حفظ شود. ولی این کار پیشرفته است و در ابتدا لازم نیست.

---

۸. جمع‌بندی

تو این مسیر، همه اجزای لازم برای ساختن یک مدل تولید شعر از صفر با پایتورچ را یاد گرفتی: از توکنایزر و دیتاست تا آموزش LSTM/Transformer و روش‌های نمونه‌برداری. می‌تونی با داده شاعر مورد نظرت شروع کنی، مدل رو بسازی و ازش مصرع‌های جدید بگیری. اگر به راه‌حل سریع‌تر و قوی‌تر نیاز داری، حتماً Fine-tuning یک GPT-2 فارسی رو امتحان کن.

هر بخش رو که خواستی عمیق‌تر توضیح بدم یا کد کامل‌تری می‌خوای، فقط بگو.